## Environments

In [ ]:
# !pip install -q librosa easydict packaging \
#                 hear21passt timm torchcodec

In [ ]:
# !pip uninstall -y mamba-ssm causal-conv1d

# !pip install causal-conv1d mamba-ssm --no-build-isolation

# !pip install flash-attn

In [ ]:
from abc import ABC, abstractmethod
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime
from os.path import exists, join
from pathlib import Path
from typing import Any, Callable, Dict, List, Literal, Optional, Tuple, Union


import pprint
import copy
import gc
import json
import shutil
import logging
import math
import nltk
import numpy as np
import os
import pandas as pd
import pickle
import plotly.express as px
import random
import re
import seaborn as sns
import tarfile
import time
import warnings
import yaml
import zipfile
import concurrent.futures

import librosa
import matplotlib.pyplot as plt
import torch
import torchaudio
import wandb
import multiprocessing as mp

from easydict import EasyDict
from dotenv import load_dotenv
# from google.colab import userdata
from huggingface_hub import login as hf_login, snapshot_download
from IPython.display import Audio, display
from nltk.corpus import wordnet
from torch import Tensor
from torch.nn.functional import dropout, linear, pad, softmax
from torch.nn.init import constant_
from torch.nn.modules.linear import Linear
from torch.nn.modules.module import Module
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm, trange
from transformers import (
    AutoConfig,
    AutoModel,
    AutoProcessor,
    AutoTokenizer,
    ClapConfig,
    ClapFeatureExtractor,
    ClapModel,
    ClapProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    pipeline,
)

import torch.nn as nn
import torch.nn.functional as F

try:
    from torch.overrides import has_torch_function, handle_torch_function
except:
    from torch._overrides import has_torch_function, handle_torch_function

from sklearn.metrics import precision_recall_curve
from scipy.optimize import linear_sum_assignment

load_dotenv()


HF_TOKEN = os.getenv("HF_TOKEN")
# HF_TOKEN = userdata.get("HF_TOKEN")
PROJECT_NAME = ""
WANDB_PROJECT_NAME = "[DCASE2026] Task6"
# WANDB_API_KEY = userdata.get("WANDB_API_KEY")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
wandb.login(key=WANDB_API_KEY)
hf_login(HF_TOKEN)

In [ ]:
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    format="%(asctime)s.%(msecs)03d:%(levelname)s:%(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)

In [ ]:
# nltk.download('wordnet', quiet=True)
# nltk.download('omw-1.4', quiet=True)

In [ ]:
# LOCAL_DIR = Path("/content/drive/MyDrive/Dataset/DCASE2026").resolve()
# DATA_DIR = LOCAL_DIR / "CLOTHO-MOMENT"

# os.listdir(DATA_DIR)

In [ ]:
# TRAIN_DIR = DATA_DIR / "train"
# VAL_DIR = DATA_DIR / "valid"
# TEST_DIR = DATA_DIR / "test"
# PREPROCESSED_DIR = DATA_DIR / "preprocessed"
# FEATURES_DIR = DATA_DIR / "features"


# print(f"Examples from train dir: {os.listdir(TRAIN_DIR)[:5]}")
# print(f"Examples from validation dir: {os.listdir(VAL_DIR)[:5]}")
# print(f"Examples from test dir: {os.listdir(TEST_DIR)[:5]}")
# print(f"Examples from pre-processed dir: {os.listdir(PREPROCESSED_DIR)[:5]}")

## Experiments


In [ ]:
def get_run_name(prefix, lr: float = 2e-5, batch_size: int = 256):
    now = datetime.now().strftime("%m%d-%H%M")
    return f"{prefix}_lr{lr}_bs{batch_size}_{now}"


get_run_name(prefix="test-run-name")

In [ ]:
def clear_gpu_cache():
    print(f"[Before] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[Before] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

    # Clear GPU cache
    torch.cuda.empty_cache()
    # Run garbage collector
    gc.collect()

    # Verify memory is cleared
    print(f"[After] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[After] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


# Verify memory is cleared
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
def set_seed(seed, use_cuda=True):
    """Sets the random seed.

    Args:
        seed (int): Seed.
        use_cuda (bool): Use cuda.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if use_cuda:
        torch.cuda.manual_seed_all(seed)

### Basic Utils

In [ ]:
class WandbLogger:
    def __init__(
        self, project_name: str, run_name: str, config: dict = None, entity: str = None
    ):
        """
        Initializes the W&B run.
        """
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=config,
            entity=entity,
            reinit=True,
        )
        self.best_accuracy = 0.0

    def log_metrics(self, metrics, step, prefix="eval"):
        """
        Logs a dictionary of metrics, converting AverageMeter to float.
        """
        log_dict = {}
        for k, v in metrics.items():
            # Extract .avg if it's an AverageMeter, otherwise keep as is
            val = v.avg if hasattr(v, "avg") else v
            log_dict[f"{prefix}/{k}"] = val

        self.run.log(log_dict, step=step)

    def log_artifact(self, model_path, name="model-checkpoint", aliases=["latest"]):
        artifact = wandb.Artifact(name, type="model")
        artifact.add_file(model_path)
        # Log with aliases like 'best' or 'production'
        self.run.log_artifact(artifact, aliases=aliases)

    def finish(self):
        """Closes the W&B run"""
        self.run.finish()

In [ ]:
def write_log(opt, epoch_i, loss_meters, metrics=None, mode="train", **kwargs):

    wandb_logger: WandbLogger = kwargs.get("wandb_logger", None)
    if wandb_logger is not None:
        wandb_logger.log_metrics(loss_meters, epoch_i, mode)
    if mode == "train":
        to_write = opt.train_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i + 1,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
        )
        filename = opt.train_log_filepath
    else:
        to_write = opt.eval_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
            eval_metrics_str=json.dumps(metrics),
        )
        filename = opt.eval_log_filepath

    with open(filename, "a") as f:
        f.write(to_write)


def save_checkpoint(model, optimizer, lr_scheduler, epoch_i, opt, **kwargs):
    wandb_logger = kwargs.get("wandb_logger", None)
    wandb_artifact_version = kwargs.get("wandb_artifact_version", [])
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "lr_scheduler": lr_scheduler.state_dict(),
        "epoch": epoch_i,
        "opt": opt,
    }
    torch.save(checkpoint, opt.ckpt_filepath)
    if wandb_logger is not None:
        wandb_logger.log_artifact(
            opt.ckpt_filepath,
            aliases=["latest"]
            if wandb_artifact_version is None
            else [wandb_artifact_version],
        )


def rename_latest_to_best(latest_file_paths):
    best_file_paths = [e.replace("latest", "best") for e in latest_file_paths]
    for src, tgt in zip(latest_file_paths, best_file_paths):
        os.renames(src, tgt)


def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_json(filename):
    with open(filename, "r") as f:
        return json.load(f)


def save_json(data, filename, save_pretty=False, sort_keys=False):
    with open(filename, "w") as f:
        if save_pretty:
            f.write(json.dumps(data, indent=4, sort_keys=sort_keys))
        else:
            json.dump(data, f)


def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(l.strip("\n")) for l in f.readlines()]


def save_jsonl(data, filename):
    """data is a list"""
    with open(filename, "w") as f:
        f.write("\n".join([json.dumps(e) for e in data]))


def save_lines(list_of_str, filepath):
    with open(filepath, "w") as f:
        f.write("\n".join(list_of_str))


def read_lines(filepath):
    with open(filepath, "r") as f:
        return [e.strip("\n") for e in f.readlines()]


def read_yaml(file_path: Union[str, Path]) -> Dict[str, Any]:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"YAML file not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = yaml.safe_load(f)
            return data if data is not None else {}
        except yaml.YAMLError as e:
            raise yaml.YAMLError(f"Error parsing YAML file {file_path}: {e}")


def mkdirp(p):
    if not os.path.exists(p):
        os.makedirs(p)


def flat_list_of_lists(l):
    """flatten a list of lists [[1,2], [3,4]] to [1,2,3,4]"""
    return [item for sublist in l for item in sublist]


def convert_to_seconds(hms_time):
    """convert '00:01:12' to 72 seconds.
    :hms_time (str): time in comma separated string, e.g. '00:01:12'
    :return (int): time in seconds, e.g. 72
    """
    times = [float(t) for t in hms_time.split(":")]
    return times[0] * 3600 + times[1] * 60 + times[2]


def get_video_name_from_url(url):
    return url.split("/")[-1][:-4]


def merge_dicts(list_dicts):
    merged_dict = list_dicts[0].copy()
    for i in range(1, len(list_dicts)):
        merged_dict.update(list_dicts[i])
    return merged_dict


def l2_normalize_np_array(np_array, eps=1e-5):
    """np_array: np.ndarray, (*, D), where the last dim will be normalized"""
    return np_array / (np.linalg.norm(np_array, axis=-1, keepdims=True) + eps)


def make_zipfile(
    src_dir,
    save_path,
    enclosing_dir="",
    exclude_dirs=None,
    exclude_extensions=None,
    exclude_dirs_substring=None,
):
    """make a zip file of root_dir, save it to save_path.
    exclude_paths will be excluded if it is a subdir of root_dir.
    An enclosing_dir is added is specified.
    """
    abs_src = os.path.abspath(src_dir)
    with zipfile.ZipFile(save_path, "w") as zf:
        for dirname, subdirs, files in os.walk(src_dir):
            if exclude_dirs is not None:
                for e_p in exclude_dirs:
                    if e_p in subdirs:
                        subdirs.remove(e_p)
            if exclude_dirs_substring is not None:
                to_rm = []
                for d in subdirs:
                    if exclude_dirs_substring in d:
                        to_rm.append(d)
                for e in to_rm:
                    subdirs.remove(e)
            arcname = os.path.join(enclosing_dir, dirname[len(abs_src) + 1 :])
            zf.write(dirname, arcname)
            for filename in files:
                if exclude_extensions is not None:
                    if os.path.splitext(filename)[1] in exclude_extensions:
                        continue  # do not zip it
                absname = os.path.join(dirname, filename)
                arcname = os.path.join(enclosing_dir, absname[len(abs_src) + 1 :])
                zf.write(absname, arcname)


class AverageMeter(object):
    """Computes and stores the average and current/max/min value"""

    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10

    def update(self, val, n=1):
        self.max = max(val, self.max)
        self.min = min(val, self.min)
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def dissect_by_lengths(np_array, lengths, dim=0, assert_equal=True):
    """Dissect an array (N, D) into a list a sub-array,
    np_array.shape[0] == sum(lengths), Output is a list of nd arrays, singlton dimention is kept"""
    if assert_equal:
        assert len(np_array) == sum(lengths)
    length_indices = [
        0,
    ]
    for i in range(len(lengths)):
        length_indices.append(length_indices[i] + lengths[i])
    if dim == 0:
        array_list = [
            np_array[length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 1:
        array_list = [
            np_array[:, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 2:
        array_list = [
            np_array[:, :, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    else:
        raise NotImplementedError
    return array_list


def get_ratio_from_counter(counter_obj, threshold=200):
    keys = counter_obj.keys()
    values = counter_obj.values()
    filtered_values = [counter_obj[k] for k in keys if k > threshold]
    return float(sum(filtered_values)) / sum(values)


def get_counter_dist(counter_object, sort_type="none"):
    _sum = sum(counter_object.values())
    dist = {k: float(f"{100 * v / _sum:.2f}") for k, v in counter_object.items()}
    if sort_type == "value":
        dist = OrderedDict(sorted(dist.items(), reverse=True))
    return dist


def get_show_name(vid_name):
    """
    get tvshow name from vid_name
    :param vid_name: video clip name
    :return: tvshow name
    """
    show_list = ["friends", "met", "castle", "house", "grey"]
    vid_name_prefix = vid_name.split("_")[0]
    show_name = vid_name_prefix if vid_name_prefix in show_list else "bbt"
    return show_name


def get_abspaths_by_ext(dir_path, ext=(".jpg",)):
    """Get absolute paths to files in dir_path with extensions specified by ext.
    Note this function does work recursively.
    """
    if isinstance(ext, list):
        ext = tuple(ext)
    if isinstance(ext, str):
        ext = tuple(
            [
                ext,
            ]
        )
    filepaths = [
        os.path.join(root, name)
        for root, dirs, files in os.walk(dir_path)
        for name in files
        if name.endswith(tuple(ext))
    ]
    return filepaths


def get_basename_no_ext(path):
    """'/data/movienet/240p_keyframe_feats/tt7672188.npz' --> 'tt7672188'"""
    return os.path.splitext(os.path.split(path)[1])[0]


def dict_to_markdown(d, max_str_len=120):
    # convert list into its str representation
    d = {k: v.__repr__() if isinstance(v, list) else v for k, v in d.items()}
    # truncate string that is longer than max_str_len
    if max_str_len is not None:
        d = {k: v[-max_str_len:] if isinstance(v, str) else v for k, v in d.items()}
    return pd.DataFrame(d, index=[0]).transpose().to_markdown()

In [ ]:
preprocessd_sample = load_jsonl(
    str(PREPROCESSED_DIR / "clotho_moment_train_release.jsonl")
)

preprocessd_sample[0]

In [ ]:
# y_sample, sr_sample = librosa.load(
#     str(TRAIN_DIR / "Venice_40_640.wav"),
#     sr=None
# )

# y_sample, sr_sample = torchaudio.load(str(TRAIN_DIR / "Venice_40_640.wav"))

In [ ]:
# display(Audio(data=y_sample,
#               rate=sr_sample))

In [ ]:
# plt.figure()
# librosa.display.waveshow(y_sample, sr=sr_sample, alpha=0.7)
# plt.title(f"Waveform – qid {preprocessd_sample[0]["qid"]}", fontsize=14, weight="bold")
# plt.xlabel("Time (s)")
# plt.ylabel("Amplitude")
# plt.tight_layout()
# plt.show()

In [ ]:
# n_fft = 1024
# hop_length = 256

# mel_spec = librosa.feature.melspectrogram(
#     y=y_sample,
#     sr=sr_sample,
#     n_fft=n_fft,
#     hop_length=hop_length,
#     n_mels=40,
#     fmax=sr_sample / 2,
# )
# log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
# plt.figure()
# librosa.display.specshow(
#     log_mel_spec,
#     sr=sr_sample,
#     hop_length=hop_length,
#     x_axis="time",
#     y_axis="mel",
#     cmap="viridis",
# )
# plt.title("Log‑Mel Spectrogram", fontsize=14, weight="bold")
# plt.colorbar(format="%+2.0f dB")
# plt.tight_layout()
# plt.show()

### Span Utils

In [ ]:
def span_xx_to_cxw(xx_spans):
    """
    Args:
        xx_spans: tensor, (#windows, 2) or (..., 2), each row is a window of format (st, ed)

    Returns:
        cxw_spans: tensor, (#windows, 2), each row is a window of format (center=(st+ed)/2, width=(ed-st))
    >>> spans = torch.Tensor([[0, 1], [0.2, 0.4]])
    >>> span_xx_to_cxw(spans)
    tensor([[0.5000, 1.0000],
        [0.3000, 0.2000]])
    >>> spans = torch.Tensor([[[0, 1], [0.2, 0.4]]])
    >>> span_xx_to_cxw(spans)
    tensor([[[0.5000, 1.0000],
         [0.3000, 0.2000]]])
    """
    center = xx_spans.sum(-1) * 0.5
    width = xx_spans[..., 1] - xx_spans[..., 0]
    return torch.stack([center, width], dim=-1)


def span_cxw_to_xx(cxw_spans):
    """
    Args:
        cxw_spans: tensor, (#windows, 2) or (..., 2), the last dim is a row denoting a window of format (center, width)

    >>> spans = torch.Tensor([[0.5000, 1.0000], [0.3000, 0.2000]])
    >>> span_cxw_to_xx(spans)
    tensor([[0.0000, 1.0000],
        [0.2000, 0.4000]])
    >>> spans = torch.Tensor([[[0.5000, 1.0000], [0.3000, 0.2000]]])
    >>> span_cxw_to_xx(spans)
    tensor([[[0.0000, 1.0000],
        [0.2000, 0.4000]]])
    """
    x1 = cxw_spans[..., 0] - 0.5 * cxw_spans[..., 1]
    x2 = cxw_spans[..., 0] + 0.5 * cxw_spans[..., 1]
    return torch.stack([x1, x2], dim=-1)


def temporal_iou(spans1, spans2):
    """
    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        iou: (N, M) torch.Tensor
        union: (N, M) torch.Tensor
    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> temporal_iou(test_spans1, test_spans2)
    (tensor([[0.6667, 0.2000],
         [0.0000, 0.5000]]),
     tensor([[0.3000, 1.0000],
             [0.8000, 1.0000]]))
    """
    areas1 = spans1[:, 1] - spans1[:, 0]  # (N, )
    areas2 = spans2[:, 1] - spans2[:, 0]  # (M, )

    left = torch.max(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.min(spans1[:, None, 1], spans2[:, 1])  # (N, M)

    inter = (right - left).clamp(min=0)  # (N, M)
    union = areas1[:, None] + areas2 - inter  # (N, M)

    iou = inter / union
    return iou, union


def temporal_intersection_over_pred(gt_spans, pred_spans):
    """intersection over the second input spans
    Args:
        gt_spans: (N, 2),
        pred_spans: (M, 2)

    Returns:

    """
    left = torch.max(gt_spans[:, None, 0], pred_spans[:, 0])
    right = torch.min(gt_spans[:, None, 1], pred_spans[:, 1])

    inter = (right - left).clamp(min=0)  # (N, M)
    inter_over_pred = inter / (pred_spans[:, 1] - pred_spans[:, 0])
    return inter_over_pred


def generalized_temporal_iou(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    assert (spans1[:, 1] >= spans1[:, 0]).all()
    assert (spans2[:, 1] >= spans2[:, 0]).all()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area


def generalized_temporal_iou_(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area

### Tensor Utils

In [ ]:
def pad_sequences_1d(
    sequences, dtype=torch.long, device=torch.device("cpu"), fixed_length=None
):
    """Pad a single-nested list or a sequence of n-d array (torch.tensor or np.ndarray)
    into a (n+1)-d array, only allow the first dim has variable lengths.
    Args:
        sequences: list(n-d tensor or list)
        dtype: np.dtype or torch.dtype
        device:
        fixed_length: pad all seq in sequences to fixed length. All seq should have a length <= fixed_length.
            return will be of shape [len(sequences), fixed_length, ...]
    Returns:
        padded_seqs: ((n+1)-d tensor) padded with zeros
        mask: (2d tensor) of the same shape as the first two dims of padded_seqs,
              1 indicate valid, 0 otherwise
    Examples:
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=torch.long)
        >>> test_data_3d = [torch.randn(2,3,4), torch.randn(4,3,4), torch.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=torch.float)
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=np.float32)
        >>> test_data_3d = [np.random.randn(2,3,4), np.random.randn(4,3,4), np.random.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=np.float32)
    """
    if isinstance(sequences[0], list):
        if "torch" in str(dtype):
            sequences = [torch.tensor(s, dtype=dtype, device=device) for s in sequences]
        else:
            sequences = [np.asarray(s, dtype=dtype) for s in sequences]

    extra_dims = sequences[0].shape[
        1:
    ]  # the extra dims should be the same for all elements
    lengths = [len(seq) for seq in sequences]
    if fixed_length is not None:
        max_length = fixed_length
    else:
        max_length = max(lengths)
    if isinstance(sequences[0], torch.Tensor):
        assert "torch" in str(dtype), "dtype and input type does not match"
        padded_seqs = torch.zeros(
            (len(sequences), max_length) + extra_dims, dtype=dtype, device=device
        )
        mask = torch.zeros(
            (len(sequences), max_length), dtype=torch.float32, device=device
        )
    else:  # np
        assert "numpy" in str(dtype), "dtype and input type does not match"
        padded_seqs = np.zeros((len(sequences), max_length) + extra_dims, dtype=dtype)
        mask = np.zeros((len(sequences), max_length), dtype=np.float32)

    for idx, seq in enumerate(sequences):
        end = lengths[idx]
        padded_seqs[idx, :end] = seq
        mask[idx, :end] = 1
    return padded_seqs, mask  # , lengths


def pad_sequences_2d(sequences, dtype=torch.long):
    """Pad a double-nested list or a sequence of n-d torch tensor into a (n+1)-d tensor,
        only allow the first two dims has variable lengths
    Args:
        sequences: list(n-d tensor or list)
        dtype: torch.long for word indices / torch.float (float32) for other cases
    Returns:
    Examples:
        >>> test_data_list = [[[1, 3, 5], [3, 7, 4, 1]], [[98, 34, 11, 89, 90], [22], [34, 56]],]
        >>> pad_sequences_2d(test_data_list, dtype=torch.long)  # torch.Size([2, 3, 5])
        >>> test_data_3d = [torch.randn(2,2,4), torch.randn(4,3,4), torch.randn(1,5,4)]
        >>> pad_sequences_2d(test_data_3d, dtype=torch.float)  # torch.Size([2, 3, 5])
        >>> test_data_3d2 = [[torch.randn(2,4), ], [torch.randn(3,4), torch.randn(5,4)]]
        >>> pad_sequences_2d(test_data_3d2, dtype=torch.float)  # torch.Size([2, 3, 5])
    # TODO add support for numpy array
    """
    bsz = len(sequences)
    para_lengths = [len(seq) for seq in sequences]
    max_para_len = max(para_lengths)
    sen_lengths = [[len(word_seq) for word_seq in seq] for seq in sequences]
    max_sen_len = max([max(e) for e in sen_lengths])

    if isinstance(sequences[0], torch.Tensor):
        extra_dims = sequences[0].shape[2:]
    elif isinstance(sequences[0][0], torch.Tensor):
        extra_dims = sequences[0][0].shape[1:]
    else:
        sequences = [
            [torch.Tensor(word_seq, dtype=dtype) for word_seq in seq]
            for seq in sequences
        ]
        extra_dims = ()

    padded_seqs = torch.zeros(
        (bsz, max_para_len, max_sen_len) + extra_dims, dtype=dtype
    )
    mask = torch.zeros(bsz, max_para_len, max_sen_len).float()

    for b_i in range(bsz):
        for sen_i, sen_l in enumerate(sen_lengths[b_i]):
            padded_seqs[b_i, sen_i, :sen_l] = sequences[b_i][sen_i]
            mask[b_i, sen_i, :sen_l] = 1
    return padded_seqs, mask  # , sen_lengths

In [ ]:
# Example

test_data_list = [[1, 2, 3], [1, 2], [3, 4, 7, 9]]
pad_sequences_1d(test_data_list, dtype=torch.long)

### Dataset

1. Prepare Dataset
2. Prepare Dataloader

In [ ]:
# --- Replaced monolithic code with imports from src/ ---
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))  # Ensure src/ is in PYTHONPATH if needed

from src.core.config import BaseOptions
from src.data.dataset import StartEndDataset, start_end_collate
from src.models.qd_detr.model import build_model
from src.pipelines.train import train_pipeline, setup_model
from src.pipelines.evaluate import evaluation_pipeline


In [ ]:
def get_dir_size(path="."):
    total_size = 0
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.is_file():
                    # entry.stat() is cached on some systems, making this very fast
                    total_size += entry.stat().st_size
                elif entry.is_dir():
                    # Recursively call the function for subdirectories
                    total_size += get_dir_size(entry.path)
    except PermissionError:
        # Handle folders you don't have access to
        return 0
    return total_size


def format_size(bytes):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if bytes < 1024:
            return f"{bytes:.2f} {unit}"
        bytes /= 1024


def _copy_single_file(src_file, root_src, dst_dir):
    """Helper function to copy a single file. (Runs inside the thread)"""
    # Calculate the relative path to maintain folder structure
    relative_path = src_file.relative_to(root_src)
    dest_item = dst_dir / relative_path

    # Ensure the destination subdirectory exists
    dest_item.parent.mkdir(parents=True, exist_ok=True)

    # Copy the file along with its metadata
    shutil.copy2(src_file, dest_item)
    return True


def copy_dir_with_progress(src, dst, max_workers=16):
    """
    Copies a directory recursively using multiple threads with a tqdm progress bar.
    """
    src_path = Path(src)
    dst_path = Path(dst)

    if not src_path.exists():
        print(f"Error: Source directory '{src}' does not exist.")
        return

    # 1. Scan and count all files
    print(f"Scanning '{src}' for files...")
    all_files = [f for f in src_path.rglob("*") if f.is_file()]
    total_files = len(all_files)

    if total_files == 0:
        print("No files found to copy.")
        return

    print(
        f"Found {total_files} files. Starting multithreaded copy with {max_workers} workers..."
    )

    # 2. Set up the Thread Pool
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all the copy tasks to the thread pool
        futures = {
            executor.submit(_copy_single_file, item, src_path, dst_path): item
            for item in all_files
        }

        # 3. Update the tqdm progress bar as each thread completes its task
        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=total_files,
            desc="Copying Data",
            unit="file",
        ):
            try:
                future.result()  # This will raise an exception if the thread failed
            except Exception as e:
                print(f"Error copying {futures[future].name}: {e}")

In [ ]:
# # # NOTE: Move dir from Drive to Colab for faster execution

# # copy_dir_with_progress(
# #     src=str(FEATURES_DIR),
# #     dst="/content"
# # )


# %timeit

# print("Copy feature dir")


# !cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/features.zip /content/

# !unzip /content/features.zip

# print("Copy pre-processed dir")

# !cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/preprocessed /content/

### Test Dataset

In [ ]:
config_path = PREPROCESSED_DIR / "train_config_clotho.yml"
opt = read_yaml(config_path)
opt

In [ ]:
!cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/preprocessed /content/

In [ ]:
opt["train_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_train.jsonl"
)
opt["val_path"] = str(
    Path("/content") / "preprocssed" / "clotho_moment_valid_val.jsonl"
)
opt["test_path"] = str(
    Path("/content") / "preprocssed" / "clotho_moment_valid_test.jsonl"
)
opt["a_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap")
opt["t_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap_text")

In [ ]:
dataset_config = EasyDict(
    data_path=opt.get("train_path"),
    ctx_mode=opt.get("ctx_mode"),
    a_feat_dir=opt.get("a_feat_dir"),
    q_feat_dir=opt.get("t_feat_dir"),
    q_feat_type="last_hidden_state",
    a_feat_type=opt.get("a_feat_type"),
    max_q_l=opt.get("max_q_l"),
    max_a_l=opt.get("max_a_l"),
    clip_len=opt.get("clip_length"),
    max_windows=opt.get("max_windows"),
    span_loss_type=opt.get("span_loss_type"),
    load_labels=True,
)

In [ ]:
train_dataset = StartEndDataset(
    **dataset_config,
)

In [ ]:
dataset_config

In [ ]:
train_sample = train_dataset[0]

train_sample

### Core Model

#### Positional Encoding

In [ ]:
# Ref: https://arxiv.org/abs/2108.12409

def get_alibi_slopes(n_heads: int):
    """Generate slopes for each head (geometric progression)"""
    start = 2 ** (-8.0 / n_heads)
    return torch.tensor(
        [start * (start**i) for i in range(n_heads)], dtype=torch.float32
    )


def create_alibi_bias(n_heads: int, seq_len: int, device: torch.device):
    """Create the ALiBi bias matrix: [n_heads, seq_len, seq_len]"""
    slopes = get_alibi_slopes(n_heads).to(device).view(n_heads, 1, 1)

    # Create distance matrix (negative for recency bias)
    pos = torch.arange(seq_len, device=device)
    distances = pos.unsqueeze(0) - pos.unsqueeze(1)  # [seq_len, seq_len]

    # Bias = -slope * distance  (only penalize past, but usually full matrix)
    alibi = distances.unsqueeze(0) * slopes  # [n_heads, seq_len, seq_len]
    return alibi  # You add this to attention scores (before softmax)


# Example integration in attention
class AttentionWithALiBi(nn.Module):
    def __init__(self, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.alibi_slopes = get_alibi_slopes(n_heads)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask=None):
        # q, k, v: [batch, heads, seq_len, head_dim]
        batch, heads, seq_len, _ = q.shape

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Add ALiBi bias
        alibi = create_alibi_bias(heads, seq_len, q.device)
        scores = scores + alibi

        if mask is not None:
            scores = scores + mask

        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)
        return output

#### Attention Module


1. Multi-head Attention


Consider

- Mamba-based Cross Attention

In [ ]:
# ==================== Saliency Guidance (from SG-DETR) ====================
import torch
import torch.nn as nn
import torch.nn.functional as F


class SaliencyGuidedCrossAttention(nn.Module):
    """Saliency-Guided Cross Attention (SGCA)"""

    def __init__(self, d_model=512, nhead=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.saliency_proj = nn.Linear(d_model, 1)  # Predict local saliency
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, audio_feat, text_feat, audio_mask=None, text_mask=None):
        """
        audio_feat: (B, L_a, D)
        text_feat:  (B, L_t, D)
        """
        # 1. Compute local saliency (how relevant each audio clip is to the text)
        # Use mean-pooled text as query
        text_pooled = text_feat.mean(dim=1, keepdim=True)  # (B, 1, D)
        local_saliency = self.saliency_proj(audio_feat)  # (B, L_a, 1)
        local_saliency = torch.sigmoid(local_saliency)  # [0, 1]

        # 2. Standard cross-attention (Audio attends to Text)
        attn_output, _ = self.cross_attn(
            query=audio_feat,
            key=text_feat,
            value=text_feat,
            key_padding_mask=~text_mask if text_mask is not None else None,
        )

        # 3. Apply saliency guidance (soft weighting)
        attn_output = attn_output * local_saliency

        # Residual + Norm
        audio_feat = self.norm(audio_feat + self.dropout(attn_output))

        return audio_feat, local_saliency.squeeze(-1)  # (B, L_a)


class SaliencyAmplifier(nn.Module):
    """Refine local saliency using global context (SG-DETR style)"""

    def __init__(self, d_model=512):
        super().__init__()
        self.global_proj = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 1)
        )

    def forward(self, audio_feat, local_saliency):
        global_context = audio_feat.mean(dim=1)  # (B, D)
        global_score = torch.sigmoid(self.global_proj(global_context)).squeeze(-1)
        refined_saliency = local_saliency * global_score.unsqueeze(1)
        return refined_saliency

#### Encoder

#### Decoder

In [ ]:
def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    if activation == "prelu":
        return nn.PReLU()
    if activation == "selu":
        return F.selu
    raise RuntimeError(f"activation should be relu/gelu, not {activation}.")


def _get_clones(module, N):
    """Creates N clones of the module.

    Args:
        module: Module to clone.
        N (int): Number of clones.

    Returns:
        nn.ModuleList: List of cloned modules.
    """
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])


def build_transformer(args):
    """Builds the transformer model.

    Args:
        args: Configuration arguments.

    Returns:
        Transformer: The transformer model.
    """
    return Transformer(
        d_model=args.hidden_dim,
        dropout=args.dropout,
        nhead=args.nheads,
        dim_feedforward=args.dim_feedforward,
        num_encoder_layers=args.enc_layers,
        num_decoder_layers=args.dec_layers,
        normalize_before=False,
        return_intermediate_dec=True,
        activation="prelu",
    )

#### Main


#### Criterion

In [ ]:
@torch.no_grad()
def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k
    output: (#items, #classes)
    target: int,
    """
    maxk = max(topk)
    num_items = output.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target)

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / num_items))
    return res

### Training and Validation


#### Computation 

#### Training pipeline


In [ ]:
train_config_dir = "/content/preprocessed/train_config_clotho.yml"
option_manager = BaseOptions(train_config_dir)
option_manager.parse()
train_opt = option_manager.option
train_opt.num_workers = 8

# --- Toggle between 'baseline' and 'enhanced' variant ---
train_opt.model_variant = "enhanced"  # Change to "baseline" to run original code

if train_opt.model_variant == "enhanced":
    train_opt.use_amp = True
    train_opt.use_focal_loss = True
    train_opt.use_flash_attention = True
    train_opt.use_compile = False  # Set to True if PyTorch 2.0+ and CUDA are available and compiled
else:
    train_opt.use_amp = False
    train_opt.use_focal_loss = False
    train_opt.use_flash_attention = False
    train_opt.use_compile = False

print(f"Selected variant: {train_opt.model_variant.upper()}")
print(f"  AMP (Mixed Precision): {train_opt.use_amp}")
print(f"  Focal Loss: {train_opt.use_focal_loss}")
print(f"  Flash Attention: {train_opt.use_flash_attention}")
print(f"  Compile: {train_opt.use_compile}")

os.makedirs("/content/results_pretraining", exist_ok=True)

train_opt.ckpt_filepath = "/content/results_pretraining/best_checkpoint.pth"
train_opt.train_log_filepath = "/content/results_pretraining/train.log"
train_opt.eval_log_filepath = "/content/results_pretraining/val.log"


In [ ]:
train_opt["train_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_train.jsonl"
)
train_opt["val_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_val.jsonl"
)
train_opt["test_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_test.jsonl"
)
train_opt["a_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap")
train_opt["t_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap_text")

In [ ]:
"""
# Updated for A100 and Long Audio
lr_drop: 100           # Drop LR halfway through 200 epochs
bsz: 2048              # Scaled for 80GB VRAM
enc_layers: 3          # Increased depth
dec_layers: 3          # Increased depth
num_queries: 20        # Better coverage for multiple events
model_ema: True        # Enabled for stability
max_a_l: 1500          # Increased to support DCASE long audio (1500s)
saliency_margin: 0.4   # Slightly wider margin for contrastive loss
"""

# train_opt.lr_drop = 100
train_opt.bsz = 3000
# train_opt.enc_layers = 3
# train_opt.dec_layers = 3
# train_opt.num_queries = 20
# train_opt.model_ema = True
# train_opt.max_a_l = 1500
# train_opt.saliency_margin = 0.4

In [ ]:
# clear_gpu_cache()

In [ ]:
# --- Inspect the model's active components ---
model, criterion, optimizer, lr_scheduler = setup_model(train_opt)
print("Focal Loss Active on Criterion:", getattr(criterion, "use_focal_loss", False))

print("
Model components verification:")
attn_types = {}
for name, module in model.named_modules():
    cls_name = module.__class__.__name__
    if "attention" in cls_name.lower() or "attn" in name or "self_attn" in name or "cross_attn" in name:
        attn_types[cls_name] = attn_types.get(cls_name, 0) + 1

for cls_name, count in attn_types.items():
    print(f"  {cls_name}: {count} occurrences")


In [ ]:
train_pipeline(train_opt, is_wandb=True)

In [ ]:
# import torch
# import gc
# from numba import cuda

# def total_gpu_cleanup():
#     # 1. Clear PyTorch references
#     gc.collect()
#     torch.cuda.empty_cache()

#     # 2. Hard reset the hardware context via Numba
#     try:
#         device = cuda.get_current_device()
#         device.reset()
#         print("Hardware context reset successful.")
#     except Exception as e:
#         print(f"Numba reset failed: {e}")

#     # 3. CRITICAL: Re-initialize PyTorch's connection to the GPU
#     # This prevents the cudaErrorInvalidValue you're seeing.
#     if torch.cuda.is_available():
#         torch.cuda.init()
#         # Create a dummy tensor to 'wake up' the driver correctly
#         _ = torch.tensor([1.0]).cuda()

#     print("PyTorch context re-initialized. Memory is fresh.")

# # Run this when you hit OOM
# total_gpu_cleanup()

#### Evaluating

In [ ]:
train_opt.model_path = "/content/results_pretraining/best_checkpoint.pth"

In [ ]:
evaluation_pipeline(opt=train_opt)

In [ ]:
LOCAL_DIR

In [ ]:
ROOT_DIR = Path("/content")
filename = "results_pretraining_11May_002"
shutil.make_archive(
    base_name=ROOT_DIR / filename,
    format="zip",
    root_dir=ROOT_DIR / "results_pretraining",
)

print(f"Archive created at {ROOT_DIR}/{filename}.zip")

In [ ]:
source = f"/content/{filename}.zip"
destination = LOCAL_DIR / "CLOTHO-MOMENT" / "pfmc" / "11-May-2026"
destination.mkdir(parents=True, exist_ok=True)

destination = destination / filename
try:
    shutil.move(source, str(destination))
    print("Move successful!")
except FileNotFoundError:
    print("Source file not found.")
except PermissionError:
    print("Permission denied.")
except Exception as e:
    print(f"An error occurred: {e}")